# Creating ways to genberate and filter PID lists from dashboard API

## generates a list of all pids between two times

In [23]:
import requests
import pandas as pd
from io import StringIO

def get_ifcb_bins_datetime_filtered(base_url: str,
                                    dataset: str,
                                    start: str,
                                    end: str,
                                    include_skip: bool = True,
                                    timeout: int = 60) -> pd.DataFrame:
    """
    Pull bins using /api/export_metadata and keep rows in [start, end].

    Parameters
    ----------
    base_url : e.g. "https://habon-ifcb.whoi.edu"
    dataset  : e.g. "nauset"
    start, end : ISO-like datetime strings, e.g. "2025-06-25T09:00:00Z"

    Returns
    -------
    DataFrame with bins in the requested time window.
    """
    base_url = base_url.rstrip("/")
    url = f"{base_url}/api/export_metadata/{dataset}"
    params = {
        "start_date": start,
        "end_date": end,
        "include_skip": str(include_skip).lower(),
    }

    r = requests.get(url, params=params, timeout=timeout)
    r.raise_for_status()

    # export_metadata returns CSV text
    df = pd.read_csv(StringIO(r.text))

    if "sample_time" not in df.columns or "pid" not in df.columns:
        raise ValueError(
            f"Expected 'sample_time' and 'pid' columns from {url}, got: {df.columns.tolist()}"
        )

    df["sample_time"] = pd.to_datetime(df["sample_time"], utc=True, errors="coerce")
    df = df.dropna(subset=["sample_time"]).sort_values("sample_time")

    start_dt = pd.to_datetime(start, utc=True)
    end_dt = pd.to_datetime(end, utc=True)

    mask = (df["sample_time"] >= start_dt) & (df["sample_time"] <= end_dt)
    return df.loc[mask].copy()


In [27]:
## example usage ##

###URL OPTIONS
## PERCY: http://percy.whoi.edu:8000
## HABON: https://habon-ifcb.whoi.edu

base_url = "https://habon-ifcb.whoi.edu"
dataset = "mvco"

start_dt = "2010-02-03T00:00:00Z"
end_dt   = "2010-02-05T12:00:00Z"


bins_df = get_ifcb_bins_datetime_filtered(base_url, dataset, start_dt, end_dt)
bins = bins_df["pid"].tolist()

print(f"Found {len(bins)} bins between {start_dt} and {end_dt}")
print(bins[:5])


Found 290 bins between 2010-02-03T00:00:00Z and 2010-02-05T12:00:00Z
['IFCB1_2010_034_000206', 'IFCB1_2010_034_001030', 'IFCB1_2010_034_002530', 'IFCB1_2010_034_003343', 'IFCB1_2010_034_005003']


In [ ]:
## To save pidlist as json file 
import json
with open("../../IFCBData/PIDLists/mdibl.json", "w") as f:  ### change the path and filename as needed
    json.dump(pids, f)

# Pick PIDS near times across date range. Let's pick just a few per day over longer timeranges

In [28]:
import requests
import pandas as pd
from io import StringIO
from typing import List, Tuple

def get_ifcb_pids_by_daily_times(
    base_url: str,
    dataset: str,
    start: str,
    end: str,
    times: List[str],
    *,
    same_day_only: bool = True,
    include_skip: bool = True,
    timeout: int = 60,
 ) -> Tuple[List[str], pd.DataFrame]:
    """
    For each day in [start, end], and for each time in `times` (HH:MM or HH:MM:SS),
    find the first bin with sample_time >= that target time.

    If same_day_only=True, matches that roll into the next day are dropped.

    Returns
    -------
    pids: list of selected pid strings (unique, in chronological order)
    picked_df: dataframe with target_time, sample_time, pid, and delta
    """
    base_url = base_url.rstrip("/")
    url = f"{base_url}/api/export_metadata/{dataset}"
    params = {
        "start_date": start,
        "end_date": end,
        "include_skip": str(include_skip).lower(),
    }

    r = requests.get(url, params=params, timeout=timeout)
    r.raise_for_status()

    # export_metadata returns CSV text
    df = pd.read_csv(StringIO(r.text))

    if "sample_time" not in df.columns or "pid" not in df.columns:
        raise ValueError(
            f"Expected 'sample_time' and 'pid' columns from {url}, got: {df.columns.tolist()}"
        )

    # Parse times
    df["sample_time"] = pd.to_datetime(df["sample_time"], utc=True, errors="coerce")
    df = df.dropna(subset=["sample_time"]).sort_values("sample_time")

    start_dt = pd.to_datetime(start, utc=True)
    end_dt = pd.to_datetime(end, utc=True)

    # Filter bins to time window (inclusive)
    df = df[(df["sample_time"] >= start_dt) & (df["sample_time"] <= end_dt)].copy()
    if df.empty:
        return [], pd.DataFrame(columns=["target_time", "sample_time", "pid", "delta"])

    # Build target schedule: every day x each desired time
    days = pd.date_range(start=start_dt.normalize(), end=end_dt.normalize(), freq="D", tz="UTC")

    # Convert "HH:MM[:SS]" strings to timedeltas
    tds = []
    for t in times:
        parts = t.split(":")
        if len(parts) == 2:
            hh, mm = parts
            ss = "0"
        elif len(parts) == 3:
            hh, mm, ss = parts
        else:
            raise ValueError(f"Time '{t}' must be 'HH:MM' or 'HH:MM:SS'")
        tds.append(pd.to_timedelta(f"{int(hh):02d}:{int(mm):02d}:{int(ss):02d}"))

    targets = pd.DataFrame({
        "target_time": [d + td for d in days for td in tds]
    }).sort_values("target_time")

    # Keep only targets that fall within the overall start/end window
    targets = targets[(targets["target_time"] >= start_dt) & (targets["target_time"] <= end_dt)].copy()
    if targets.empty:
        return [], pd.DataFrame(columns=["target_time", "sample_time", "pid", "delta"])

    # Forward "asof" join: for each target_time, find first sample_time >= target_time
    bins_for_asof = df[["sample_time", "pid"]].rename(columns={"sample_time": "sample_time_match"})
    bins_for_asof = bins_for_asof.sort_values("sample_time_match")

    picked = pd.merge_asof(
        targets.sort_values("target_time"),
        bins_for_asof,
        left_on="target_time",
        right_on="sample_time_match",
        direction="forward",
        allow_exact_matches=True,
    )

    picked = picked.rename(columns={"sample_time_match": "sample_time"})
    picked = picked.dropna(subset=["sample_time", "pid"]).copy()

    # Optionally enforce same-day matches
    if same_day_only:
        picked = picked[picked["sample_time"].dt.normalize() == picked["target_time"].dt.normalize()].copy()

    picked["delta"] = picked["sample_time"] - picked["target_time"]

    # Optional: de-duplicate if two targets land on the same PID (happens if bins are sparse)
    picked = picked.sort_values("target_time")
    picked_unique = picked.drop_duplicates(subset=["pid"], keep="first").copy()

    pids = picked_unique["pid"].tolist()
    return pids, picked_unique[["target_time", "sample_time", "pid", "delta"]]

In [29]:
### Example Usage ###
pids, picked_df = get_ifcb_pids_by_daily_times(
    base_url="https://habon-ifcb.whoi.edu",
    dataset="mvco",
    start="2017-01-01T00:00:00Z",
    end="2017-06-25T23:59:59Z",
    times=["00:00", "06:00", "12:00", "18:00"],   # pick ~4 per da or change as needed
    same_day_only=True
)

print(pids[:10])
print(picked_df.head(10))
print(len(pids))

['IFCB5_2017_001_001057', 'IFCB5_2017_001_061312', 'IFCB5_2017_001_121620', 'IFCB5_2017_001_180423', 'IFCB5_2017_002_001539', 'IFCB5_2017_002_060343', 'IFCB5_2017_002_122101', 'IFCB5_2017_002_180904', 'IFCB5_2017_003_151126', 'IFCB5_2017_003_181638']
                 target_time               sample_time                    pid  \
0  2017-01-01 00:00:00+00:00 2017-01-01 00:10:57+00:00  IFCB5_2017_001_001057   
1  2017-01-01 06:00:00+00:00 2017-01-01 06:13:12+00:00  IFCB5_2017_001_061312   
2  2017-01-01 12:00:00+00:00 2017-01-01 12:16:20+00:00  IFCB5_2017_001_121620   
3  2017-01-01 18:00:00+00:00 2017-01-01 18:04:23+00:00  IFCB5_2017_001_180423   
4  2017-01-02 00:00:00+00:00 2017-01-02 00:15:39+00:00  IFCB5_2017_002_001539   
5  2017-01-02 06:00:00+00:00 2017-01-02 06:03:43+00:00  IFCB5_2017_002_060343   
6  2017-01-02 12:00:00+00:00 2017-01-02 12:21:01+00:00  IFCB5_2017_002_122101   
7  2017-01-02 18:00:00+00:00 2017-01-02 18:09:04+00:00  IFCB5_2017_002_180904   
8  2017-01-03 00:00:

In [30]:
print(len(pids),len(picked_df))

529 529


In [22]:
## To save pidlist as json file 
import json
with open("../../IFCBData/PIDLists/mvco_test.json", "w") as f:
    json.dump(pids, f)